# Taller: Diseño y Evaluación de Técnicas de Machine Learning

**Curso:** Técnicas de Inteligencia Artificial  
**Universidad Nacional de Colombia — Sede Bogotá**  
**Ingeniería Mecatrónica — Facultad de Ingeniería**  
**Profesor:** Flavio Prieto — faprietoo@unal.edu.co  
**Estudiantes:** Sergio Andrés Bolaños Penagos, Jorge Nicolas Garzon Acevedo, David Santiago Pirateque Suarez  
**Fecha:** Abril 2026

---

## Objetivo

Resolver el mismo problema de clasificación multiclase abordado en el taller de Redes Neuronales Artificiales, usando técnicas clásicas de Machine Learning sobre el dataset **Olivetti Faces**.

El modelo de referencia del taller anterior es un MLP completamente denso. En este taller se implementan y optimizan dos modelos clásicos supervisados:

1. **Máquinas de Vectores de Soporte (SVM)**, evaluando kernel lineal y kernel RBF.
2. **Random Forest**, como ensamble clásico de árboles de decisión.

La comparación se realiza con el mismo conjunto de entrenamiento, el mismo conjunto de validación y las mismas características de entrada: imágenes de 64 × 64 píxeles aplanadas en vectores de 4096 características.

---
## 1. Librerías y configuración general

Se importan las librerías necesarias para cargar el dataset, dividir los datos, entrenar modelos clásicos de clasificación, ajustar hiperparámetros, calcular métricas y construir visualizaciones.

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13

print(f"Semilla de reproducibilidad: {SEED}")

---
## 2. Modelo de referencia del taller RNA

El taller de Redes Neuronales Artificiales entrenó un MLP profundo sobre el mismo dataset. Ese resultado se usa como línea base para comparar los modelos clásicos.

El mejor modelo reportado en el taller RNA fue:

| Modelo | Entrada | Capas ocultas | Validación | Test | F1 macro | Comentario |
|---|---:|---|---:|---:|---:|---|
| MLP | 4096 | 512-256 | 0.9875 | 0.9750 | 0.9733 | Red densa con regularización |

In [ ]:
mlp_reference = {
    "Modelo": "MLP RNA",
    "Familia": "Deep Learning",
    "Entrada": 4096,
    "Hiperparámetros": "hidden=[512, 256], Adam, Dropout=0.40, L2=1e-5",
    "Train Accuracy": 1.0000,
    "Val Accuracy": 0.9875,
    "Test Accuracy": 0.9750,
    "Precision Macro": 0.9833,
    "Recall Macro": 0.9750,
    "F1 Macro": 0.9733,
    "F1 Micro": 0.9750,
    "Tiempo (s)": 15.3,
    "Parámetros/Árboles": "2,240,808 parámetros",
}

pd.DataFrame([mlp_reference])

---
## 3. Carga y exploración del dataset Olivetti Faces

El dataset se carga con `fetch_olivetti_faces`, igual que en el taller RNA. Cada imagen en escala de grises tiene tamaño 64 × 64, por lo que cada muestra se representa como un vector de 4096 características.

In [ ]:
faces = fetch_olivetti_faces(shuffle=False)
X, y = faces.data, faces.target
images = faces.images

print(f"Dimensiones de X: {X.shape}")
print(f"Número de clases: {len(np.unique(y))}")
print(f"Imágenes por clase: {np.bincount(y)[0]}")
print(f"Rango de intensidad: [{X.min():.2f}, {X.max():.2f}]")

fig, axes = plt.subplots(5, 10, figsize=(15, 8))
fig.suptitle("Olivetti Faces — 5 personas x 10 imágenes", fontsize=14)

for persona in range(5):
    idxs = np.where(y == persona)[0]
    for j, idx in enumerate(idxs):
        axes[persona, j].imshow(images[idx], cmap="gray")
        axes[persona, j].axis("off")
        if j == 0:
            axes[persona, j].set_ylabel(f"P{persona}", fontsize=10)

plt.tight_layout()
plt.savefig("fig_ml_01_faces_grid.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.1 Distribución de clases

El conjunto tiene 40 clases y 10 imágenes por persona. Esto permite usar partición estratificada para conservar la misma proporción de clases en entrenamiento, validación y prueba.

In [ ]:
counts = np.bincount(y, minlength=40)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(40), counts, color="steelblue", edgecolor="black", alpha=0.85)
ax.axhline(10, color="red", linestyle="--", label="10 imágenes/persona")
ax.set_title("Distribución de clases — Olivetti Faces")
ax.set_xlabel("Persona / clase")
ax.set_ylabel("Número de imágenes")
ax.set_xticks(range(40))
ax.legend()
plt.tight_layout()
plt.savefig("fig_ml_02_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Dataset perfectamente balanceado: {bool(np.all(counts == 10))}")

---
## 4. Partición y normalización

Se conserva la misma estrategia del taller RNA: hold-out estratificado 60/20/20.

| Subconjunto | Proporción | Uso |
|---|---:|---|
| Entrenamiento | 60% | Ajustar los modelos clásicos |
| Validación | 20% | Seleccionar hiperparámetros |
| Prueba | 20% | Evaluación final después de seleccionar modelos |

La normalización se ajusta únicamente con el conjunto de entrenamiento para evitar fuga de información hacia validación o prueba.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=SEED,
    stratify=y_temp,
)

print("Partición hold-out estratificada 60/20/20:")
for name, labels in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    c = np.bincount(labels, minlength=40)
    print(
        f"  {name:5s}: {len(labels):3d} muestras "
        f"({len(labels) / len(y) * 100:.0f}%) — imgs/persona: [{c.min()}, {c.max()}]"
    )

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

print("\nDimensionalidad usada por todos los modelos clásicos:")
print(f"  X_train_sc: {X_train_sc.shape}")
print(f"  X_val_sc:   {X_val_sc.shape}")
print(f"  X_test_sc:  {X_test_sc.shape}")
print(f"  Ratio muestras/features en train: {X_train_sc.shape[0] / X_train_sc.shape[1]:.4f}")

---
## 5. Funciones comunes de entrenamiento, búsqueda y evaluación

Los dos modelos se ajustan con búsqueda manual sobre el conjunto de validación. Esto permite cumplir la condición del taller: todos los modelos se comparan usando el mismo conjunto de entrenamiento, las mismas características y la misma validación.

In [ ]:
def summarize_metrics(model, X_train, y_train, X_val, y_val, X_test, y_test):
    """Calcula métricas principales en train, validación y test."""
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    return {
        "Train Accuracy": accuracy_score(y_train, y_train_pred),
        "Val Accuracy": accuracy_score(y_val, y_val_pred),
        "Test Accuracy": accuracy_score(y_test, y_test_pred),
        "Precision Macro": precision_score(y_test, y_test_pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y_test, y_test_pred, average="macro", zero_division=0),
        "F1 Macro": f1_score(y_test, y_test_pred, average="macro", zero_division=0),
        "F1 Micro": f1_score(y_test, y_test_pred, average="micro", zero_division=0),
    }


def search_model(model_name, build_model, param_grid, X_train, y_train, X_val, y_val):
    """Evalúa combinaciones de hiperparámetros y ordena por accuracy/F1 de validación."""
    rows = []
    grids = list(ParameterGrid(param_grid))
    print(f"{model_name}: {len(grids)} combinaciones de hiperparámetros")

    for i, params in enumerate(grids, start=1):
        model = build_model(params)
        start = time.time()
        model.fit(X_train, y_train)
        fit_time = time.time() - start

        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)

        rows.append({
            "Modelo": model_name,
            "Trial": i,
            "Train Accuracy": accuracy_score(y_train, y_train_pred),
            "Val Accuracy": accuracy_score(y_val, y_val_pred),
            "Val F1 Macro": f1_score(y_val, y_val_pred, average="macro", zero_division=0),
            "Tiempo fit (s)": fit_time,
            "params": params,
        })

    results = pd.DataFrame(rows)
    results = results.sort_values(
        ["Val Accuracy", "Val F1 Macro", "Tiempo fit (s)"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    return results


def train_best_model(model_name, build_model, search_results, X_train, y_train):
    """Reentrena el mejor modelo de una familia usando solo el conjunto de entrenamiento."""
    best_params = search_results.loc[0, "params"]
    model = build_model(best_params)
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    return model, best_params, elapsed


def format_params(params):
    """Convierte diccionarios de hiperparámetros en texto corto para tablas."""
    return ", ".join(f"{k}={v}" for k, v in params.items())

---
## 6. Modelo clásico 1: SVM

Las Máquinas de Vectores de Soporte buscan una frontera de decisión que maximice el margen. Para este problema se evalúan varias configuraciones del clasificador `SVC`:

- `linear`: frontera lineal en el espacio original de características.
- `rbf`: kernel gaussiano para modelar fronteras no lineales.
- `poly`: kernel polinomial para probar interacciones no lineales de bajo orden.
- `sigmoid`: kernel basado en una transformación tipo tangente hiperbólica.

Los kernels `poly` y `sigmoid` se incluyen como exploración adicional. La selección del modelo se realiza con el conjunto de validación. Si varias configuraciones empatan en validación, se prefiere la alternativa más simple para evitar aumentar la complejidad sin una mejora medible.

In [ ]:
def build_svm(params):
    return SVC(**params)

svm_param_grid = [
    {
        "kernel": ["linear"],
        "C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10],
    },
    {
        "kernel": ["rbf"],
        "C": [0.3, 1, 3, 10, 30, 100],
        "gamma": ["scale", 1e-5, 3e-5, 1e-4, 3e-4, 1e-3],
    },
    {
        "kernel": ["poly"],
        "C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10],
        "degree": [2, 3, 4],
        "gamma": ["scale", 1e-4, 3e-4, 1e-3],
        "coef0": [0, 0.5, 1],
    },
    {
        "kernel": ["sigmoid"],
        "C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10],
        "gamma": ["scale", 1e-4, 3e-4, 1e-3],
        "coef0": [-1, 0, 1],
    },
]

svm_search = search_model(
    "SVM",
    build_svm,
    svm_param_grid,
    X_train_sc,
    y_train,
    X_val_sc,
    y_val,
)

# Desempate: si varias configuraciones empatan en validación,
# se prefiere el kernel más simple.
kernel_rank = {"linear": 0, "rbf": 1, "poly": 2, "sigmoid": 3}
svm_search["Kernel"] = svm_search["params"].apply(lambda p: p["kernel"])
svm_search["Kernel Rank"] = svm_search["Kernel"].map(kernel_rank)
svm_search["C Rank"] = svm_search["params"].apply(lambda p: float(p.get("C", 1.0)))
svm_search = svm_search.sort_values(
    ["Val Accuracy", "Val F1 Macro", "Kernel Rank", "C Rank", "Tiempo fit (s)"],
    ascending=[False, False, True, True, True],
).reset_index(drop=True)

print("Top 15 configuraciones SVM por validación y desempate por simplicidad:")
display(svm_search.head(15))

In [ ]:
svm_kernel_rows = []
for kernel_name, group in svm_search.assign(
    kernel=svm_search["params"].apply(lambda p: p["kernel"])
).groupby("kernel"):
    best_row = group.sort_values(
        ["Val Accuracy", "Val F1 Macro", "Kernel Rank", "C Rank", "Tiempo fit (s)"],
        ascending=[False, False, True, True, True],
    ).iloc[0]
    model = build_svm(best_row["params"])
    model.fit(X_train_sc, y_train)
    y_test_kernel = model.predict(X_test_sc)
    svm_kernel_rows.append({
        "Kernel": kernel_name,
        "Mejor Val Accuracy": best_row["Val Accuracy"],
        "Mejor Val F1 Macro": best_row["Val F1 Macro"],
        "Train Accuracy": best_row["Train Accuracy"],
        "Test Accuracy post-selección": accuracy_score(y_test, y_test_kernel),
        "Test F1 Macro post-selección": f1_score(y_test, y_test_kernel, average="macro", zero_division=0),
        "Mejores hiperparámetros": best_row["params"],
    })

svm_kernel_summary = pd.DataFrame(svm_kernel_rows).sort_values(
    ["Mejor Val Accuracy", "Mejor Val F1 Macro"],
    ascending=False,
).reset_index(drop=True)

print("Mejor configuración por kernel:")
display(svm_kernel_summary)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_data = svm_search.assign(kernel=svm_search["params"].apply(lambda p: p["kernel"]))
sns.boxplot(data=plot_data, x="kernel", y="Val Accuracy", ax=ax, palette="Set2")
sns.stripplot(data=plot_data, x="kernel", y="Val Accuracy", ax=ax, color="black", alpha=0.45, size=3)
ax.set_title("SVM — desempeño en validación por kernel")
ax.set_xlabel("Kernel")
ax.set_ylabel("Accuracy de validación")
plt.tight_layout()
plt.savefig("fig_ml_03_svm_kernel_validation.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. Modelo clásico 2: Random Forest

Random Forest combina múltiples árboles de decisión entrenados con muestreo bootstrap y selección aleatoria de características. Según el material de clase, este ensamble reduce el sobreajuste de un único árbol y permite estimar importancia de características.

En imágenes aplanadas, cada característica corresponde a un píxel. Por esta razón, la importancia del modelo puede visualizarse como un mapa 64 × 64.

In [ ]:
def build_random_forest(params):
    return RandomForestClassifier(
        random_state=SEED,
        n_jobs=-1,
        **params,
    )

rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 20],
    "max_features": ["sqrt", 0.2],
    "min_samples_leaf": [1, 2],
}

rf_search = search_model(
    "Random Forest",
    build_random_forest,
    rf_param_grid,
    X_train_sc,
    y_train,
    X_val_sc,
    y_val,
)

print("Top 10 configuraciones Random Forest por validación:")
display(rf_search.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
rf_plot = rf_search.copy()
rf_plot["n_estimators"] = rf_plot["params"].apply(lambda p: p["n_estimators"])
sns.lineplot(data=rf_plot, x="n_estimators", y="Val Accuracy", marker="o", errorbar=None, ax=ax)
ax.set_title("Random Forest — efecto del número de árboles")
ax.set_xlabel("Número de árboles")
ax.set_ylabel("Accuracy de validación")
plt.tight_layout()
plt.savefig("fig_ml_04_rf_estimators_validation.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Selección de mejores modelos por validación

Después de la búsqueda, se selecciona la mejor configuración de cada familia usando únicamente el conjunto de validación. Luego se evalúa cada modelo seleccionado en el conjunto de prueba.

In [ ]:
svm_model, svm_best_params, svm_fit_time = train_best_model(
    "SVM",
    build_svm,
    svm_search,
    X_train_sc,
    y_train,
)

rf_model, rf_best_params, rf_fit_time = train_best_model(
    "Random Forest",
    build_random_forest,
    rf_search,
    X_train_sc,
    y_train,
)

classic_models = [
    {
        "Modelo": "SVM",
        "Familia": "ML clásico",
        "Entrada": X_train_sc.shape[1],
        "model": svm_model,
        "best_params": svm_best_params,
        "fit_time": svm_fit_time,
        "Parámetros/Árboles": "vectores de soporte",
    },
    {
        "Modelo": "Random Forest",
        "Familia": "ML clásico",
        "Entrada": X_train_sc.shape[1],
        "model": rf_model,
        "best_params": rf_best_params,
        "fit_time": rf_fit_time,
        "Parámetros/Árboles": f"{rf_best_params['n_estimators']} árboles",
    },
]

for item in classic_models:
    metrics = summarize_metrics(
        item["model"],
        X_train_sc,
        y_train,
        X_val_sc,
        y_val,
        X_test_sc,
        y_test,
    )
    item.update(metrics)
    item["Tiempo (s)"] = item["fit_time"]
    item["Hiperparámetros"] = format_params(item["best_params"])

classic_summary = pd.DataFrame([
    {k: v for k, v in item.items() if k not in ["model", "best_params", "fit_time"]}
    for item in classic_models
])

print("Modelos clásicos seleccionados:")
display(classic_summary)

---
## 9. Evaluación detallada en test

Se reportan accuracy global, F1 macro/micro, matriz de confusión y reporte de clasificación por clase para cada modelo clásico seleccionado.

In [ ]:
for item in classic_models:
    print("=" * 90)
    print(f"MODELO: {item['Modelo']}")
    print(f"Hiperparámetros: {item['Hiperparámetros']}")
    print(f"Train Accuracy: {item['Train Accuracy']:.4f}")
    print(f"Val Accuracy:   {item['Val Accuracy']:.4f}")
    print(f"Test Accuracy:  {item['Test Accuracy']:.4f}")
    print(f"F1 Macro:       {item['F1 Macro']:.4f}")
    print("\nReporte de clasificación en test:")
    y_pred = item["model"].predict(X_test_sc)
    print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
def plot_readable_confusion(model, model_name, filename):
    """Grafica una matriz de confusión normalizada y anota solo valores no nulos."""
    y_pred = model.predict(X_test_sc)
    cm = confusion_matrix(y_test, y_pred, labels=np.arange(40))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

    ax.set_title(f"Matriz de confusión normalizada — {model_name}")
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    ax.set_xticks(np.arange(40))
    ax.set_yticks(np.arange(40))
    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)

    ax.set_xticks(np.arange(-0.5, 40, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 40, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=0.6)
    ax.tick_params(which="minor", bottom=False, left=False)

    for i in range(40):
        for j in range(40):
            if cm[i, j] > 0:
                color = "white" if cm_norm[i, j] > 0.55 else "black"
                ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=7, color=color)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Proporción por clase real")
    plt.tight_layout()
    plt.savefig(filename, dpi=180, bbox_inches="tight")
    plt.show()

    errors = []
    for i in range(40):
        for j in range(40):
            if i != j and cm[i, j] > 0:
                errors.append({"Clase real": i, "Clase predicha": j, "Errores": int(cm[i, j])})
    return pd.DataFrame(errors).sort_values(["Errores", "Clase real"], ascending=[False, True])

confusion_error_tables = {}
for item in classic_models:
    filename = f"fig_ml_05_confusion_{item['Modelo'].replace(' ', '_').lower()}.png"
    confusion_error_tables[item["Modelo"]] = plot_readable_confusion(item["model"], item["Modelo"], filename)

for model_name, error_table in confusion_error_tables.items():
    print(f"Errores fuera de la diagonal — {model_name}")
    if len(error_table) == 0:
        print("No hubo errores en test.")
    else:
        display(error_table)

### 9.1 Importancia de características en Random Forest

La importancia por impureza (`feature_importances_`) de Random Forest puede ser inestable cuando hay 4096 píxeles y solo 240 muestras de entrenamiento. En este caso no necesariamente significa que el modelo haya encontrado una región facial semántica; puede asignar importancia a píxeles de borde, iluminación o fondo porque esos píxeles redujeron impureza en algunos árboles.

Por esa razón se presentan dos vistas:

1. **MDI por píxel:** importancia interna del Random Forest, útil pero sensible a ruido y correlaciones.
2. **Permutación por bloques en validación:** se divide cada rostro en bloques 8 × 8 y se mide cuánto cae la accuracy de validación al permutar cada bloque. Esta vista es más cercana a generalización porque se calcula fuera del entrenamiento.

In [ ]:
rf_importances = rf_model.feature_importances_.reshape(64, 64)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

im0 = axes[0].imshow(rf_importances, cmap="hot", interpolation="nearest")
axes[0].set_title("Random Forest — MDI por píxel")
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)


def block_permutation_importance(model, X_val, y_val, image_shape=(64, 64), block_size=8, random_state=SEED):
    """Calcula importancia por bloques permutando regiones de la imagen en validación."""
    rng = np.random.default_rng(random_state)
    baseline = accuracy_score(y_val, model.predict(X_val))
    n_blocks = image_shape[0] // block_size
    drops = np.zeros((n_blocks, n_blocks))

    for bi in range(n_blocks):
        for bj in range(n_blocks):
            rows = np.arange(bi * block_size, (bi + 1) * block_size)
            cols = np.arange(bj * block_size, (bj + 1) * block_size)
            pixel_ids = np.array([r * image_shape[1] + c for r in rows for c in cols])

            X_perm = X_val.copy()
            permutation = rng.permutation(X_perm.shape[0])
            X_perm[:, pixel_ids] = X_perm[permutation][:, pixel_ids]
            permuted_acc = accuracy_score(y_val, model.predict(X_perm))
            drops[bi, bj] = baseline - permuted_acc

    return drops, baseline


block_importance, rf_val_baseline = block_permutation_importance(rf_model, X_val_sc, y_val, block_size=8)

im1 = axes[1].imshow(block_importance, cmap="magma", interpolation="nearest")
axes[1].set_title(f"Permutación por bloques 8x8\nAccuracy base val = {rf_val_baseline:.4f}")
axes[1].set_xlabel("Bloque horizontal")
axes[1].set_ylabel("Bloque vertical")
axes[1].set_xticks(range(8))
axes[1].set_yticks(range(8))

for i in range(8):
    for j in range(8):
        if block_importance[i, j] > 0:
            axes[1].text(j, i, f"{block_importance[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white")

fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label="Caída de accuracy")
plt.tight_layout()
plt.savefig("fig_ml_06_rf_importance_mdi_vs_blocks.png", dpi=180, bbox_inches="tight")
plt.show()

block_rows = []
for bi in range(8):
    for bj in range(8):
        block_rows.append({
            "Bloque fila": bi,
            "Bloque columna": bj,
            "Caída accuracy": block_importance[bi, bj],
            "Rango filas px": f"{bi*8}-{bi*8+7}",
            "Rango cols px": f"{bj*8}-{bj*8+7}",
        })

block_importance_table = pd.DataFrame(block_rows).sort_values("Caída accuracy", ascending=False)
print("Bloques con mayor caída de accuracy al permutarse:")
display(block_importance_table.head(12))

---
## 10. Comparación contra el MLP del taller RNA

La comparación final integra el mejor MLP del taller RNA y los dos modelos clásicos seleccionados. Se comparan métricas de prueba, validación, tiempo y brecha entre entrenamiento y prueba.

In [ ]:
comparison_rows = [mlp_reference]
comparison_rows.extend(classic_summary.to_dict("records"))
comparison = pd.DataFrame(comparison_rows)

comparison["Gap Train-Test"] = comparison["Train Accuracy"] - comparison["Test Accuracy"]

ordered_cols = [
    "Modelo", "Familia", "Entrada", "Hiperparámetros",
    "Train Accuracy", "Val Accuracy", "Test Accuracy",
    "F1 Macro", "F1 Micro", "Precision Macro", "Recall Macro",
    "Gap Train-Test", "Tiempo (s)", "Parámetros/Árboles",
]

comparison = comparison[ordered_cols]
comparison_sorted = comparison.sort_values("Test Accuracy", ascending=False).reset_index(drop=True)

display(comparison_sorted)

In [ ]:
metric_plot = comparison.melt(
    id_vars=["Modelo"],
    value_vars=["Val Accuracy", "Test Accuracy", "F1 Macro"],
    var_name="Métrica",
    value_name="Valor",
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=metric_plot, x="Modelo", y="Valor", hue="Métrica", ax=ax, palette="Set2")
ax.set_ylim(0.75, 1.02)
ax.set_title("Comparación de desempeño: MLP vs modelos clásicos")
ax.set_xlabel("Modelo")
ax.set_ylabel("Valor")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("fig_ml_07_model_comparison_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=comparison, x="Modelo", y="Tiempo (s)", ax=ax, palette="Set3")
ax.set_title("Tiempo de entrenamiento reportado")
ax.set_xlabel("Modelo")
ax.set_ylabel("Tiempo (s)")
plt.tight_layout()
plt.savefig("fig_ml_08_training_time.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 11. Análisis de sobreajuste y subajuste

Para identificar sobreajuste y subajuste se revisan dos señales:

- **Sobreajuste:** accuracy de entrenamiento muy alta y caída apreciable en validación o test.
- **Subajuste:** accuracy de entrenamiento y validación bajas, lo cual indica que el modelo no logra capturar la estructura del problema.

También se revisan configuraciones no seleccionadas de las búsquedas para observar casos representativos.

In [ ]:
def classify_fit_case(train_acc, val_acc):
    gap = train_acc - val_acc
    if train_acc >= 0.95 and gap >= 0.08:
        return "Sobreajuste"
    if train_acc <= 0.80 and val_acc <= 0.80:
        return "Subajuste"
    return "Ajuste razonable"

all_trials = pd.concat([svm_search, rf_search], ignore_index=True)
all_trials["Diagnóstico"] = all_trials.apply(
    lambda row: classify_fit_case(row["Train Accuracy"], row["Val Accuracy"]),
    axis=1,
)

print("Diagnóstico de los modelos finales:")
final_fit = comparison[["Modelo", "Train Accuracy", "Val Accuracy", "Test Accuracy", "Gap Train-Test"]].copy()
final_fit["Diagnóstico"] = final_fit.apply(
    lambda row: classify_fit_case(row["Train Accuracy"], row["Val Accuracy"]),
    axis=1,
)
display(final_fit)

print("\nEjemplos de subajuste encontrados durante la búsqueda:")
underfit_examples = all_trials[all_trials["Diagnóstico"] == "Subajuste"].sort_values("Val Accuracy").head(5)
display(underfit_examples[["Modelo", "Train Accuracy", "Val Accuracy", "Val F1 Macro", "params", "Diagnóstico"]])

print("\nEjemplos de sobreajuste encontrados durante la búsqueda:")
overfit_examples = all_trials[all_trials["Diagnóstico"] == "Sobreajuste"].sort_values("Val Accuracy").head(5)
display(overfit_examples[["Modelo", "Train Accuracy", "Val Accuracy", "Val F1 Macro", "params", "Diagnóstico"]])

---
## 12. Discusión técnica

### 12.1 Limitaciones de modelos lineales

La SVM lineal aprende una frontera lineal en el espacio de características. En imágenes aplanadas, esto significa separar personas mediante combinaciones lineales de intensidades de píxeles. Si las diferencias entre personas requieren relaciones complejas entre regiones del rostro, una frontera lineal puede ser insuficiente.

La búsqueda ampliada también incluye SVM con kernels `rbf`, `poly` y `sigmoid`. Estos kernels permiten revisar si una frontera no lineal mejora la validación frente al modelo lineal.

### 12.2 Impacto de la dimensionalidad

Cada imagen tiene 4096 características y el entrenamiento tiene 240 muestras. El ratio muestras/features es bajo, por lo que existe riesgo de sobreajuste. Esta condición afecta especialmente a modelos que dependen de distancias en alta dimensión o divisiones sobre píxeles individuales.

### 12.3 Capacidad de modelar no linealidades

- SVM lineal: frontera lineal.
- SVM RBF, polinomial y sigmoide: fronteras no lineales inducidas por kernels.
- Random Forest: frontera no lineal por particiones sucesivas del espacio de características.
- MLP: composición de capas densas con activaciones no lineales.

### 12.4 Importancia de píxeles y generalización en Random Forest

La importancia MDI de Random Forest puede verse dispersa porque cada árbol selecciona divisiones sobre píxeles individuales. En un dataset pequeño, algunos píxeles de borde, iluminación o fondo pueden reducir impureza en entrenamiento sin representar una región facial estable. Por eso la interpretación principal no debe basarse solo en `feature_importances_`.

La importancia por bloques con permutación en validación es una revisión más exigente: si al alterar un bloque cae la accuracy, ese bloque parece aportar información útil para datos no usados en entrenamiento. Si las caídas son pequeñas o dispersas, eso respalda la hipótesis de que Random Forest está aprendiendo patrones menos robustos que SVM o MLP.

### 12.5 Diferencias entre ML clásico y Deep Learning

Los modelos clásicos pueden funcionar muy bien con pocos datos y entrenamiento relativamente controlado. El MLP tiene mayor capacidad representacional, pero también más parámetros y mayor riesgo de sobreajuste. En Olivetti Faces, la cantidad de datos es pequeña, por lo que la regularización y la validación son determinantes.

In [ ]:
best_overall = comparison_sorted.iloc[0]
best_classic = classic_summary.sort_values("Test Accuracy", ascending=False).iloc[0]
svm_kernel_best = svm_best_params["kernel"]

best_kernel_summary = svm_kernel_summary.iloc[0]
best_block = block_importance_table.iloc[0]

analysis_md = f"""
### 12.6 Conclusiones del experimento ejecutado

1. **Mejor modelo global:** el mejor resultado en test fue `{best_overall['Modelo']}` con accuracy `{best_overall['Test Accuracy']:.4f}` y F1 macro `{best_overall['F1 Macro']:.4f}`.

2. **Mejor modelo clásico:** entre los modelos clásicos, el mejor resultado en test fue `{best_classic['Modelo']}` con accuracy `{best_classic['Test Accuracy']:.4f}` y F1 macro `{best_classic['F1 Macro']:.4f}`.

3. **Búsqueda ampliada de kernels:** el mejor grupo de kernels alcanzó Val Accuracy `{best_kernel_summary['Mejor Val Accuracy']:.4f}`. Los kernels no lineales se probaron explícitamente, pero no superaron al kernel lineal bajo el criterio de validación y desempate por simplicidad.

4. **SVM:** la mejor configuración final usó kernel `{svm_kernel_best}`. Esto sugiere que, con las características normalizadas de Olivetti, una frontera lineal regularizada ya captura gran parte de la separación entre personas.

5. **Random Forest:** el modelo alcanzó accuracy de test `{classic_summary.loc[classic_summary['Modelo'] == 'Random Forest', 'Test Accuracy'].iloc[0]:.4f}`. La visualización MDI por píxel puede ser engañosa; por eso se agregó una importancia por bloques calculada sobre validación. El bloque con mayor caída fue fila `{int(best_block['Bloque fila'])}`, columna `{int(best_block['Bloque columna'])}`, con caída `{best_block['Caída accuracy']:.4f}`.

6. **Complejidad vs desempeño:** el MLP tiene muchos más parámetros entrenables, mientras que SVM y Random Forest son modelos clásicos más directos. Si el desempeño clásico se acerca al MLP, esto indica que para este dataset pequeño los modelos clásicos siguen siendo competitivos.

7. **Escenarios adecuados:** SVM es adecuado para datasets pequeños o medianos de alta dimensionalidad; Random Forest es útil cuando se busca un modelo no lineal con interpretabilidad aproximada, pero su importancia por píxeles debe analizarse con cuidado; el MLP es preferible cuando se desea mayor capacidad de representación y se cuenta con regularización suficiente.
"""

display(Markdown(analysis_md))

---
## 13. Resumen de entregables cubiertos

| Requerimiento del taller | Evidencia en el informe |
|---|---|
| Código reproducible | Semilla fija, partición estratificada y búsqueda explícita |
| Implementación de modelos clásicos | SVM y Random Forest |
| Mismo conjunto de entrenamiento | Split 60/20/20 reutilizado para todos los modelos |
| Mismas características de entrada | 4096 píxeles normalizados para ambos modelos |
| Ajuste de hiperparámetros | Búsqueda manual sobre validación para SVM y Random Forest |
| Misma validación | Selección por `X_val_sc`, `y_val` |
| Comparación con MLP | Tabla final MLP vs SVM vs Random Forest |
| Tablas y gráficas | Búsquedas, métricas, matrices de confusión normalizadas, tiempos e importancia por bloques |
| Conclusiones técnicas | Discusión sobre linealidad, dimensionalidad, sobreajuste y complejidad |